In [1]:

import numpy as np
import heapq
import pandas as pd
from sklearn.cluster import KMeans
import os


In [2]:

# =====================================================
# PARAMETERS
# =====================================================

SIM_TIME             = 45 * 24
SPEED                = 20
AREA_SIZE            = 20
BASE_FARE            = 3
FARE_PER_MILE        = 2
PETROL_COST_PER_MILE = 0.20
CVAR_ALPHA           = 0.05
PATIENCE_RATE        = 5

LAM_VALUES = np.linspace(1, 4.9, 40).tolist()
K_VALUES   = range(4, 5)

OUTPUT_DIR = "Comparison_4_Idea_June"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [3]:

# =====================================================
# GIVEN vs MLE PARAMS
# =====================================================

GIVEN = {
    'driver_rate' : 3,
    'rider_rate'  : 30,
    'driver_loc'  : lambda: np.array([np.random.uniform(0, AREA_SIZE),
                                      np.random.uniform(0, AREA_SIZE)]),
    'pickup_loc'  : lambda: np.array([np.random.uniform(0, AREA_SIZE),
                                      np.random.uniform(0, AREA_SIZE)]),
    'dropoff_loc' : lambda: np.array([np.random.uniform(0, AREA_SIZE),
                                      np.random.uniform(0, AREA_SIZE)]),
}

MLE = {
    'driver_rate' : 4.74,
    'rider_rate'  : 34.6,
    'driver_loc'  : lambda: np.array([
        np.clip(np.random.normal(9.9739,  4.3647), 0, AREA_SIZE),
        np.clip(np.random.normal(11.5133, 4.3362), 0, AREA_SIZE)]),
    'pickup_loc'  : lambda: np.array([
        np.clip(np.random.normal(8.3597,  4.2571), 0, AREA_SIZE),
        np.clip(np.random.normal(12.3175, 4.1904), 0, AREA_SIZE)]),
    'dropoff_loc' : lambda: np.array([
        np.clip(np.random.normal(11.2297, 4.5390), 0, AREA_SIZE),
        np.clip(np.random.normal(13.2626, 4.1690), 0, AREA_SIZE)]),
}

PARAM_SETS = {'given': GIVEN, 'mle': MLE}


In [4]:

# =====================================================
# KMEANS HUB CONFIGS
# =====================================================

def parse_coord(s):
    s = str(s).strip('()').split(',')
    return float(s[0]), float(s[1])

print("Fitting KMeans hubs...")
riders = pd.read_excel('riders.xlsx')
riders[['px','py']] = pd.DataFrame(
    riders['pickup_location'].apply(parse_coord).tolist(), index=riders.index)
X = riders[['px','py']].values

HUB_CONFIGS = {}
for k in range(1, 11):
    km    = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    km.fit(X)
    sizes = np.bincount(km.labels_)
    order = np.argsort(-sizes)
    HUB_CONFIGS[k] = km.cluster_centers_[order]
print("Done.\n")


Fitting KMeans hubs...
Done.



In [5]:

# =====================================================
# DRIVER & RIDER
# =====================================================

class Driver:
    def __init__(self, did, loc, ou):
        self.id=did; self.location=loc; self.online_until=ou
        self.earnings=0; self.status='idle'; self.busy_time=0

class Rider:
    def __init__(self, rid, o, d, arr, ab):
        self.id=rid; self.origin=o; self.destination=d
        self.arrival_time=arr; self.abandon_time=ab; self.matched=False


In [6]:

# =====================================================
# SIMULATION — Idea 3b, spec-correct
# =====================================================

class Simulation:

    def __init__(self, params, hubs, lam):
        self.params = params
        self.hubs   = hubs
        self.lam    = lam

        self.current_time     = 0
        self.event_list       = []
        self.idle_drivers     = {}
        self.busy_drivers     = {}
        self.waiting_riders   = {}
        self.driver_counter   = 0
        self.rider_counter    = 0
        self.total_riders     = 0
        self.abandoned_riders = 0
        self.total_queue_wait = 0
        self.total_true_wait  = 0
        self.driver_log       = []

    def exp_time(self, r): return np.random.exponential(1 / r)
    def dist(self, a, b):  return np.linalg.norm(np.array(a) - np.array(b))
    def ttime(self, d):
        mu = d / SPEED; return np.random.uniform(0.8 * mu, 1.2 * mu)
    def schedule(self, t, e, d=None):
        heapq.heappush(self.event_list, (t, e, d))

    def nearest_hub_dist(self, loc):
        return min(np.linalg.norm(loc - h) for h in self.hubs)

    def _raw_scores(self, driver):
        """Compute raw s1 (urgency) and s2 (hub) for every waiting rider."""
        raw = {}
        for rid, rider in self.waiting_riders.items():
            d2p                = self.dist(driver.location, rider.origin)
            remaining_patience = max(rider.abandon_time - self.current_time, 1e-6)
            s1 = d2p / remaining_patience                                # urgency
            s2 = d2p + self.lam * self.nearest_hub_dist(rider.destination)  # hub
            raw[rid] = (s1, s2)
        return raw

    def _best_rider(self, driver):
        """  return rider with lowest Score_new = score1/score2."""
        raw = self._raw_scores(driver)

        def new_score(rid):
            s1, s2   = raw[rid]
            scaled = s1 / s2 
            return scaled  # GH's idea

        return min(raw, key=new_score)

    # ── Spec-correct matching ────────────────────────────────────────
    def try_match(self, triggered_by, entity_id):
        """
        triggered_by : 'rider'  -> rider just arrived  -> find closest driver to this rider
                       'driver' -> driver just freed up -> find best rider via June score
        entity_id    : the rid or did that triggered this call
        """
        if not self.idle_drivers or not self.waiting_riders:
            return

        if triggered_by == 'rider':
            # Rider arrived — find the closest DRIVER to this rider
            rider    = self.waiting_riders[entity_id]
            best_did = min(self.idle_drivers,
                           key=lambda d: self.dist(self.idle_drivers[d].location,
                                                   rider.origin))
            best_rid = entity_id

        else:  # triggered_by == 'driver'
            # Driver freed up — find best RIDER using GH combined score
            best_rid = self._best_rider(self.idle_drivers[entity_id])  # ← fixed
            best_did = entity_id

        driver = self.idle_drivers[best_did]
        rider  = self.waiting_riders[best_rid]
        rider.matched = True

        pd_ = self.dist(driver.location, rider.origin)
        td  = self.dist(rider.origin, rider.destination)
        pt  = self.ttime(pd_); tt2 = self.ttime(td); tt = pt + tt2

        qw = self.current_time - rider.arrival_time
        self.total_queue_wait += qw
        self.total_true_wait  += qw + pt

        earn = (BASE_FARE + FARE_PER_MILE * td) - \
               PETROL_COST_PER_MILE * (pd_ + td)

        driver.status = 'busy'; driver.busy_time += tt
        self.busy_drivers[best_did] = driver
        del self.idle_drivers[best_did]; del self.waiting_riders[best_rid]
        self.schedule(self.current_time + tt, 'DROPOFF',
                      (best_did, rider.destination, earn))

    def handle_driver_arrival(self):
        self.driver_counter += 1; did = self.driver_counter
        loc = self.params['driver_loc']()
        ou  = self.current_time + np.random.uniform(6, 8)
        self.idle_drivers[did] = Driver(did, loc, ou)
        self.driver_log.append({'driver_id': did, 'arrival_time': self.current_time,
                                'online_until': ou, 'earnings': 0, 'busy_time': 0})
        self.schedule(ou, 'DRIVER_OFFLINE', did)
        self.schedule(self.current_time + self.exp_time(self.params['driver_rate']),
                      'DRIVER_ARRIVAL')
        self.try_match(triggered_by='driver', entity_id=did)

    def handle_rider_arrival(self):
        self.rider_counter += 1; rid = self.rider_counter
        self.total_riders += 1
        o  = self.params['pickup_loc']()
        d  = self.params['dropoff_loc']()
        ab = self.current_time + self.exp_time(PATIENCE_RATE)
        self.waiting_riders[rid] = Rider(rid, o, d, self.current_time, ab)
        self.schedule(ab, 'RIDER_ABANDON', rid)
        self.schedule(self.current_time + self.exp_time(self.params['rider_rate']),
                      'RIDER_ARRIVAL')
        self.try_match(triggered_by='rider', entity_id=rid)

    def handle_dropoff(self, data):
        did, nloc, earn = data; driver = self.busy_drivers[did]
        driver.earnings += earn; driver.location = nloc; driver.status = 'idle'
        for d in self.driver_log:
            if d['driver_id'] == did:
                d['earnings'] += earn; d['busy_time'] = driver.busy_time; break
        del self.busy_drivers[did]
        if self.current_time < driver.online_until:
            self.idle_drivers[did] = driver
            self.try_match(triggered_by='driver', entity_id=did)

    def run(self):
        self.schedule(0, 'DRIVER_ARRIVAL'); self.schedule(0, 'RIDER_ARRIVAL')
        while self.event_list and self.current_time < SIM_TIME:
            self.current_time, et, data = heapq.heappop(self.event_list)
            if   et == 'DRIVER_ARRIVAL': self.handle_driver_arrival()
            elif et == 'RIDER_ARRIVAL':  self.handle_rider_arrival()
            elif et == 'DROPOFF':        self.handle_dropoff(data)
            elif et == 'RIDER_ABANDON':
                if data in self.waiting_riders:
                    self.abandoned_riders += 1; del self.waiting_riders[data]
            elif et == 'DRIVER_OFFLINE':
                self.idle_drivers.pop(data, None)
        return self.collect_results()

    def collect_results(self):
        matched = self.total_riders - self.abandoned_riders
        df      = pd.DataFrame(self.driver_log)
        df['online_duration']  = df['online_until'] - df['arrival_time']
        df['busy_time_capped'] = df[['busy_time','online_duration']].min(axis=1)
        df['idle_time']        = df['online_duration'] - df['busy_time_capped']
        df['idle_ratio']       = df['idle_time'] / df['online_duration']

        p  = df['earnings'].values; N = len(p); mp = p.mean()
        vp = np.sum((p - mp)**2) / (N - 1) if N > 1 else 0
        ps = np.sort(p)
        cvar     = np.mean(ps[:max(1, int(np.floor(CVAR_ALPHA * N)))])
        cvar_med = np.mean(ps[:max(1, int(np.floor(0.5 * N)))])

        return {
            'abandonment_rate'   : self.abandoned_riders / self.total_riders,
            'avg_queue_wait'     : self.total_queue_wait / matched if matched > 0 else 0,
            'avg_true_wait'      : self.total_true_wait  / matched if matched > 0 else 0,
            'matched_riders'     : matched,
            'total_riders'       : self.total_riders,
            'abandoned_riders'   : self.abandoned_riders,
            'avg_earnings'       : mp,
            'total_earnings'     : df['earnings'].sum(),
            'avg_profit_per_hr'  : df['earnings'].sum() / df['online_duration'].sum(),
            'avg_idle_time'      : df['idle_time'].mean(),
            'avg_idle_ratio'     : df['idle_ratio'].mean(),
            'variance'           : vp,
            'fairness_cv'        : np.sqrt(vp) / mp if mp > 0 else 0,
            'cvar'               : cvar,
            'delta_cvar_internal': cvar - cvar_med,
            '_driver_df'         : df,
        }


In [7]:

# =====================================================
# BASELINES — one per param set
# =====================================================

class BaselineSim(Simulation):
    def try_match(self, triggered_by, entity_id):
        if not self.idle_drivers or not self.waiting_riders: return
        if triggered_by == 'rider':
            rider    = self.waiting_riders[entity_id]
            best_did = min(self.idle_drivers,
                           key=lambda d: self.dist(self.idle_drivers[d].location,
                                                   rider.origin))
            best_rid = entity_id
        else:
            driver   = self.idle_drivers[entity_id]
            best_rid = min(self.waiting_riders,
                           key=lambda r: self.dist(driver.location,
                                                   self.waiting_riders[r].origin))
            best_did = entity_id

        driver = self.idle_drivers[best_did]
        rider  = self.waiting_riders[best_rid]
        rider.matched = True

        pd_ = self.dist(driver.location, rider.origin)
        td  = self.dist(rider.origin, rider.destination)
        pt  = self.ttime(pd_); tt2 = self.ttime(td); tt = pt + tt2

        qw = self.current_time - rider.arrival_time
        self.total_queue_wait += qw
        self.total_true_wait  += qw + pt

        earn = (BASE_FARE + FARE_PER_MILE * td) - \
               PETROL_COST_PER_MILE * (pd_ + td)

        driver.status = 'busy'; driver.busy_time += tt
        self.busy_drivers[best_did] = driver
        del self.idle_drivers[best_did]; del self.waiting_riders[best_rid]
        self.schedule(self.current_time + tt, 'DROPOFF',
                      (best_did, rider.destination, earn))


In [8]:

# =====================================================
# RUN BASELINES
# =====================================================

baselines = {}
for pname, params in PARAM_SETS.items():
    print(f"Running baseline [{pname}]...")
    np.random.seed(42)
    res = BaselineSim(params, hubs=np.array([[0,0]]), lam=0).run()
    baselines[pname] = res
    res['_driver_df'].to_csv(f"{OUTPUT_DIR}/data_baseline_{pname}.csv", index=False)
    print(f"  Abandon: {res['abandonment_rate']*100:.2f}%  Earn: £{res['avg_earnings']:.2f}")
print()


Running baseline [given]...


  Abandon: 22.40%  Earn: £157.66
Running baseline [mle]...
  Abandon: 1.49%  Earn: £116.20



In [9]:

# =====================================================
# MAIN LOOP — param_set x lambda x K
# =====================================================

summary_rows = []
for pname, b in baselines.items():
    summary_rows.append({
        'param_set': pname, 'lambda': 0.0, 'n_hubs': 0,
        'config': f'baseline_{pname}',
        'abandonment_pct'   : b['abandonment_rate'] * 100,
        'avg_queue_wait_min': b['avg_queue_wait'] * 60,
        'avg_true_wait_min' : b['avg_true_wait']  * 60,
        'matched_riders'    : b['matched_riders'],
        'avg_earnings'      : b['avg_earnings'],
        'total_earnings'    : b['total_earnings'],
        'avg_profit_per_hr' : b['avg_profit_per_hr'],
        'avg_idle_ratio'    : b['avg_idle_ratio'],
        'variance'          : b['variance'],
        'cvar'              : b['cvar'],
        'fairness_effect'   : 0.0,
        'delta_cvar_cross'  : 0.0,
        'delta_profit'      : 0.0,
    })

total_runs = len(PARAM_SETS) * len(LAM_VALUES) * len(list(K_VALUES))
run_num = 0

for pname, params in PARAM_SETS.items():
    b  = baselines[pname]
    bv = b['variance']
    bc = b['cvar']

    print(f"\n{'='*70}")
    print(f"  PARAM SET: {pname.upper()}")
    print(f"{'='*70}")

    for lam in LAM_VALUES:
        lam_rows = []
        print(f"\n  λ={lam}")
        print(f"  {'Config':<22} {'Abandon%':>9} {'AvgEarn£':>9} {'FairnessEff':>12} {'ΔCVaR£':>9}")
        print(f"  {'-'*65}")

        for k in K_VALUES:
            run_num += 1
            config = f"{pname}_lam{lam}_k{k}"

            np.random.seed(42)
            sim = Simulation(params, hubs=HUB_CONFIGS[k], lam=lam)
            res = sim.run()

            fe           = 1 - res['variance'] / bv if bv > 0 else 0
            delta_cvar_x = res['cvar'] - bc
            delta_profit = res['avg_earnings'] - b['avg_earnings']

            row = {
                'param_set'         : pname,
                'lambda'            : lam,
                'n_hubs'            : k,
                'config'            : config,
                'abandonment_pct'   : res['abandonment_rate'] * 100,
                'avg_queue_wait_min': res['avg_queue_wait'] * 60,
                'avg_true_wait_min' : res['avg_true_wait']  * 60,
                'matched_riders'    : res['matched_riders'],
                'avg_earnings'      : res['avg_earnings'],
                'total_earnings'    : res['total_earnings'],
                'avg_profit_per_hr' : res['avg_profit_per_hr'],
                'avg_idle_ratio'    : res['avg_idle_ratio'],
                'variance'          : res['variance'],
                'cvar'              : res['cvar'],
                'fairness_effect'   : fe,
                'delta_cvar_cross'  : delta_cvar_x,
                'delta_profit'      : delta_profit,
            }
            summary_rows.append(row)
            lam_rows.append(row)

            print(f"  {config:<22} {res['abandonment_rate']*100:>8.2f}% "
                  f"{res['avg_earnings']:>9.2f} {fe:>+12.4f} "
                  f"{delta_cvar_x:>+9.2f}  [{run_num}/{total_runs}]")

        # save per param_set per lambda
        lam_str = f"{lam:.1f}".replace(".", "_")
        pd.DataFrame(lam_rows).to_csv(
            f"{OUTPUT_DIR}/data_{pname}_lam{lam_str}.csv", index=False)
        print(f"  -> Saved: data_{pname}_lam{lam_str}.csv")

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(f"{OUTPUT_DIR}/summary.csv", index=False)
print(f"\n{OUTPUT_DIR}/summary.csv")


  PARAM SET: GIVEN

  λ=1.0
  Config                  Abandon%  AvgEarn£  FairnessEff    ΔCVaR£
  -----------------------------------------------------------------
  given_lam1.0_k4           26.56%    147.39      +0.0121    -13.96  [1/80]
  -> Saved: data_given_lam1_0.csv

  λ=1.1
  Config                  Abandon%  AvgEarn£  FairnessEff    ΔCVaR£
  -----------------------------------------------------------------
  given_lam1.1_k4           26.23%    147.80      -0.0000    -11.51  [2/80]
  -> Saved: data_given_lam1_1.csv

  λ=1.2
  Config                  Abandon%  AvgEarn£  FairnessEff    ΔCVaR£
  -----------------------------------------------------------------
  given_lam1.2_k4           23.78%    147.63      -0.0156    -11.22  [3/80]
  -> Saved: data_given_lam1_2.csv

  λ=1.3
  Config                  Abandon%  AvgEarn£  FairnessEff    ΔCVaR£
  -----------------------------------------------------------------
  given_lam1.3_k4           29.37%    147.03      +0.0255    -11.88  [